# 3. CamemBERT : un Transformer qui comprend le contexte

**Objectif** : fine-tuner CamemBERT (un modèle pré-entraîné sur du français) pour classer les avis Allociné, puis le comparer à TF-IDF (93,8 %).

### Avant de commencer (dans Google Colab)
1. **Exécution → Modifier le type d'exécution → GPU T4**, puis Enregistrer. Sans GPU, l'entraînement prendrait des heures.
2. Exécute les cellules **dans l'ordre** avec Maj + Entrée.
3. Garde l'onglet ouvert pendant l'entraînement : Colab peut couper une session inactive, et tout ce qui est en mémoire est alors perdu.

In [ ]:
# Installe les bibliothèques (déjà présentes sur Colab en général, ça ne coûte rien de vérifier)
!pip install -q transformers datasets accelerate sentencepiece

In [ ]:
import torch

print("GPU disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Carte :", torch.cuda.get_device_name(0))
else:
    print("⚠️ Pas de GPU : active-le dans Exécution > Modifier le type d'exécution")

## 3.1 Charger les données

On les télécharge directement depuis Hugging Face : pas besoin d'envoyer tes fichiers sur Colab.

**On n'entraîne que sur 20 000 avis** (au lieu de 160 000) : sur un GPU gratuit, ça prend environ 5 à 10 minutes au lieu d'une heure. On évalue sur **toute la validation** (20 000 avis), la même que pour TF-IDF, pour que la comparaison soit juste.

Remarque : TF-IDF a été entraîné sur 160 000 avis, CamemBERT n'en verra que 20 000, soit **8 fois moins**. Si CamemBERT fait quand même mieux, c'est grâce à ce qu'il a appris avant (le pré-entraînement).

In [ ]:
from datasets import load_dataset

N_TRAIN = 20_000

ds = load_dataset("tblard/allocine")
train_petit = ds["train"].shuffle(seed=42).select(range(N_TRAIN))
val = ds["validation"]

print(train_petit)
print(val)

## 3.2 La tokenisation en sous-mots

TF-IDF découpait le texte en **mots entiers**. CamemBERT découpe en **sous-mots** (*tokens*) :
- les mots fréquents restent entiers (`film`) ;
- les mots rares sont coupés en morceaux connus (`ininregardable` → `▁in` + `in` + `regard` + `able`).

**Pourquoi ?** Le vocabulaire est limité (32 000 tokens), mais avec des morceaux on peut écrire **n'importe quel mot**, même une faute de frappe ou un mot jamais vu. Et le modèle peut deviner le sens d'un mot rare grâce à ses morceaux (`in-` = négation).

Le symbole `▁` indique un début de mot. Chaque token a un **numéro** (son identifiant dans le vocabulaire) : c'est cette liste de numéros que lit le modèle.

In [ ]:
from transformers import AutoTokenizer

NOM_MODELE = "almanach/camembert-base"
tokenizer = AutoTokenizer.from_pretrained(NOM_MODELE)

phrase = "Un navet ininregardable, mais pas mauvais du tout."
print(tokenizer.tokenize(phrase))
print(tokenizer(phrase)["input_ids"])

### À TOI (3.2)
1. Tokenise ton **avis piège n°2** (« Film pas mauvais, qui se laisse regarder… ») avec `tokenizer.tokenize(...)`.
2. Compare le **nombre de tokens** (`len(...)`) au **nombre de mots** (`len(texte.split())`). Pourquoi y a-t-il plus de tokens que de mots ?
3. La liste `input_ids` commence et finit par deux numéros spéciaux. Affiche les tokens correspondants avec `tokenizer.convert_ids_to_tokens(ids)`. Que sont `<s>` et `</s>` à ton avis ?

In [ ]:
# À TOI


**Mes réponses (3.2)** : ...

## 3.3 Tokeniser tout le jeu de données

CamemBERT ne peut lire que **512 tokens au maximum**. On coupe à **256** (`truncation=True, max_length=256`) : c'est environ deux fois plus rapide. Mais une partie des avis sera coupée : tu vas mesurer combien juste après.

`.map(..., batched=True)` applique la fonction à tous les avis, par paquets, très rapidement.

In [ ]:
def tokeniser(batch):
    return tokenizer(batch["review"], truncation=True, max_length=256)

train_tok = train_petit.map(tokeniser, batched=True)
val_tok = val.map(tokeniser, batched=True)
train_tok

### À TOI (3.3)
Vérifie notre choix de 256 : **quel pourcentage des avis de validation dépasse 256 tokens** (et sera donc coupé) ?

*Indice* : tokenise les 2 000 premiers avis de `val["review"]` **sans** `truncation`, calcule la longueur de chaque liste `input_ids`, puis la proportion supérieure à 256. (Rappel : `(liste_de_longueurs > 256).mean()` marche si tu mets les longueurs dans un tableau NumPy ou une Series pandas.)

In [ ]:
# À TOI


**Question** : la troncature garde le **début** de l'avis et coupe la **fin**. Pense à ton avis piège n°1 (« … Verdict : c'est sympa… mais très lourd »). Pourquoi couper la fin d'une critique peut-il être un problème ?

**Ma réponse (3.3)** : ...

## 3.4 Le modèle

`AutoModelForSequenceClassification` charge CamemBERT **et ajoute au-dessus une petite couche de décision** à 2 sorties (négatif, positif).

- Le corps de CamemBERT (110 millions de paramètres) est **déjà entraîné** : il comprend le français.
- La couche de décision est **neuve**, avec des poids au hasard : c'est elle qui va surtout apprendre.

Un message d'avertissement va dire que certains poids sont « newly initialized » : **c'est normal**, c'est justement cette nouvelle couche.

In [ ]:
from transformers import AutoModelForSequenceClassification

modele = AutoModelForSequenceClassification.from_pretrained(NOM_MODELE, num_labels=2)
print(f"{modele.num_parameters():,} paramètres")

## 3.5 L'entraînement (fine-tuning)

On utilise le `Trainer` de Hugging Face, qui gère la boucle d'entraînement pour nous. Les réglages importants :

| Paramètre | Valeur | Signification |
|---|---|---|
| `num_train_epochs` | 1 | le modèle voit chaque avis d'entraînement **une fois** |
| `per_device_train_batch_size` | 16 | il apprend par paquets de 16 avis |
| `learning_rate` | 2e-5 | **très petit** : on ajuste doucement un modèle qui sait déjà beaucoup, sans lui faire « oublier » le français |
| `fp16` | True | calculs en demi-précision : environ 2 fois plus rapide sur GPU |

Le **`DataCollatorWithPadding`** complète les avis courts avec des tokens vides pour que tous les avis d'un paquet aient la même longueur (un GPU calcule sur des tableaux rectangulaires).

`calculer_accuracy` dit au Trainer comment noter le modèle : le modèle sort deux scores par avis (les *logits*), on prend le plus grand avec `argmax`, et on compare aux vraies réponses.

⏱️ **Durée attendue : 5 à 10 minutes** sur un GPU T4. La barre de progression s'affiche pendant l'entraînement.

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score
from transformers import DataCollatorWithPadding, Trainer, TrainingArguments


def calculer_accuracy(eval_pred):
    logits, vraies_reponses = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(vraies_reponses, predictions)}


reglages = TrainingArguments(
    output_dir="camembert-allocine",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    fp16=torch.cuda.is_available(),
    eval_strategy="no",      # on évaluera nous-mêmes après l'entraînement
    save_strategy="no",      # pas de sauvegarde intermédiaire (on sauvegarde à la fin)
    logging_steps=100,
    report_to="none",
)

trainer = Trainer(
    model=modele,
    args=reglages,
    train_dataset=train_tok,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=calculer_accuracy,
)

trainer.train()

## 3.6 À TOI : évaluer sur la validation

1. Utilise `trainer.evaluate(val_tok)` pour obtenir l'accuracy sur les 20 000 avis de validation (≈ 1 à 2 minutes).
2. Complète le tableau de comparaison ci-dessous.
3. Calcule la **réduction d'erreurs** par rapport à TF-IDF + bigrammes. Rappel : erreur = 1 − accuracy. Par exemple, passer de 6 % à 3 % d'erreurs, c'est 50 % d'erreurs en moins.

In [ ]:
# À TOI


| Modèle | Données d'entraînement | Accuracy (validation) |
|---|---|---|
| TF-IDF + régression logistique | 160 000 avis | 92,5 % |
| TF-IDF + bigrammes | 160 000 avis | 93,8 % |
| **CamemBERT** | 20 000 avis | ... |

**Réduction d'erreurs par rapport aux bigrammes** : ...

**Ma conclusion** : ...

## 3.7 À TOI : tes avis pièges

Écris une fonction `predire(texte)` qui renvoie la probabilité que l'avis soit positif.

*Les étapes* :
1. `entrees = tokenizer([texte], return_tensors="pt", truncation=True, max_length=256).to(modele.device)` : tokenise et envoie sur le GPU ;
2. `with torch.no_grad():` puis `logits = modele(**entrees).logits` : `no_grad` dit à PyTorch qu'on ne s'entraîne pas, ce qui économise de la mémoire ;
3. `probas = torch.softmax(logits, dim=-1)` : le **softmax** transforme les 2 scores en 2 probabilités qui somment à 1. C'est la généralisation de la sigmoïde de la régression logistique ;
4. renvoie `probas[0, 1].item()`, la probabilité de « positif ».

Puis teste tes **3 avis pièges** (« génial » qui parle d'un autre film, « Film pas mauvais… », et ton avis inventé « Un film d'un grand intérêt, pas mauvais du tout. »).

**Question** : CamemBERT fait-il mieux que TF-IDF sur ces avis ? Pourquoi, d'après ce que tu sais de l'attention ?

In [ ]:
# À TOI


| Avis | Vraie réponse | TF-IDF + bigrammes (proba positif) | CamemBERT (proba positif) |
|---|---|---|---|
| Piège 1 (« génial ») | négatif | 22 % | ... |
| Piège 2 (« pas mauvais ») | positif | 38 % | ... |
| Piège 3 (inventé) | positif | 0,5 % | ... |

**Ma conclusion** : ...

## 3.8 Sauvegarder le modèle dans Google Drive

⚠️ **Tout ce qui est sur Colab disparaît à la fin de la session.** On sauvegarde le modèle dans ton Google Drive pour le réutiliser à l'étape 5 (démo). Colab va te demander l'autorisation d'accéder à ton Drive.

Le modèle pèse environ 450 Mo.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

dossier = "/content/drive/MyDrive/camembert-allocine"
trainer.save_model(dossier)
tokenizer.save_pretrained(dossier)
print("Modèle sauvegardé dans", dossier)